# 01 Baseline Replay
Thin notebook: orchestration only. Business logic lives in src/.

In [6]:
import os
import sys
import importlib
from IPython.display import display

cwd = os.getcwd()
candidates = [cwd, os.path.dirname(cwd)]
REPO = next((p for p in candidates if os.path.exists(os.path.join(p, 'src'))), cwd)
SRC = os.path.join(REPO, 'src')
if SRC not in sys.path:
    sys.path.insert(0, SRC)

import margin_analysis
importlib.reload(margin_analysis)
import charts
importlib.reload(charts)
import replay_analyzer
importlib.reload(replay_analyzer)
from replay_analyzer import default_analyzer_for_repo

In [7]:
analyzer = default_analyzer_for_repo(REPO)
baseline = analyzer.replay_folder(os.path.join(REPO, 'data', 'baseline'), export_dir=os.path.join(REPO, 'output'))

print('Detected Baseline Configuration')
display(baseline.configuration())

print('Summary Metrics')
display(baseline.summary())

print('Per Pair Statistics')
display(baseline.pair_table())

No XLSX found for EURGBP, skipping
No XLSX found for GBPUSD, skipping
No XLSX found for USDCHF, skipping


Detected Baseline Configuration


,Pair,Risk,TP,Grid,Max Trades,Trades,Initial Balance,Median Lot
0,CADJPY,1.4,15,60,3,309,5000.0,0.09
1,EURUSD,1.2,23,29,4,575,5000.0,0.10
2,NZDCHF,1.8,10,31,5,315,5000.0,0.12


Summary Metrics


,Initial Balance,Final Balance,Profit,Average Monthly Growth,CAGR,Max Floating Drawdown %,Peak-to-Trough Equity Drawdown %,Minimum Margin Level,Minimum Free Margin,Profit Factor,Recovery Factor,Total Trades,Winning Trades,Win Rate
0,5000.0,34316.9754,29316.9754,4.132916,64.8189,-35.684529,29.368895,137.4286,2176.4973,2.683277,5.462242,2398,890,37.114262


Per Pair Statistics


,Pair,Risk,Baseline Risk,Take Profit,Grid,Max Trades,Trades,Scaled PnL
0,CADJPY,1.4,1.4,15,60,3,309,3889.6830
1,EURUSD,1.2,1.2,23,29,4,575,20655.8366
2,NZDCHF,1.8,1.8,10,31,5,315,4771.4557


In [8]:
import pandas as pd
import margin_analysis
import charts

ma = baseline.margin_analysis()
events = ma.events_df()
threshold_summary = ma.threshold_summary_df(events)

print('Portfolio Margin Summary')
display(pd.DataFrame([ma.portfolio_summary_dict(events)]).T.rename(columns={0: 'value'}))

print('\nThreshold Summary (Canonical Events)')
display(threshold_summary)

print('\nPer-Pair Contribution Summary')
display(ma.pair_summary_df(events))

if not events.empty:
    print(f'\nCanonical Margin Events ({len(events)} episodes) — worst 10 by margin level')
    display(events.nsmallest(10, 'min_margin_level')[
        ['event_id', 'start_time', 'end_time', 'duration_minutes',
         'min_margin_level', 'worst_threshold_crossed', 'thresholds_crossed',
         'max_used_margin', 'largest_used_margin_contributor',
         'largest_used_margin_contribution_pct']
    ])

    print('\nBasket Composition at 3 Worst Events')
    display(ma.event_baskets_df(events.nsmallest(3, 'min_margin_level')))

Portfolio Margin Summary


,value
min_margin_level,137.43
avg_margin_level,1043.12
median_margin_level,615.42
p95_used_margin,6210.08
max_used_margin,19806.43
max_free_margin,34420.37
min_free_margin,2176.50
total_minutes_below_300,229775.00
event_count_below_300,110.00
longest_event_minutes_below_300,86340.00



Threshold Summary (Canonical Events)


,threshold,event_count,total_minutes_below,longest_event_minutes_below,worst_margin_level
0,300.0,110,229775,86340,137.43
1,200.0,14,161921,86340,137.43
2,150.0,2,33363,31065,137.43
3,100.0,0,0,0,NaN



Per-Pair Contribution Summary


,pair,avg_used_margin,max_used_margin,avg_used_margin_contribution_pct,peak_used_margin_contribution_pct,avg_floating_pnl,worst_floating_pnl,avg_open_positions,max_open_positions,margin_events_participated,margin_events_as_largest_contributor
0,CADJPY,1050.50,5131.78,34.68,100.0,-436.24,-3304.41,1.38,3,100,34
1,EURUSD,469.11,13940.70,15.47,100.0,-32.59,-5201.74,1.10,4,99,45
2,NZDCHF,1101.40,3142.61,48.23,100.0,-382.74,-3408.85,2.20,5,110,31



Canonical Margin Events (110 episodes) — worst 10 by margin level


,event_id,start_time,end_time,duration_minutes,min_margin_level,worst_threshold_crossed,thresholds_crossed,max_used_margin,largest_used_margin_contributor,largest_used_margin_contribution_pct
46,evt_1712153520_47,2024-04-03 14:12:00,2024-04-05 04:30:00,2298,137.43,150.0,"300,200,150",5815.07,EURUSD,50.49
54,evt_1715920200_55,2024-05-17 04:30:00,2024-06-07 18:15:00,31065,148.21,150.0,"300,200,150",5279.93,EURUSD,37.65
9,evt_1702996860_10,2023-12-19 14:41:00,2023-12-20 22:45:00,1924,154.40,200.0,"300,200",5589.94,EURUSD,71.07
109,evt_1790182440_110,2026-09-23 16:54:00,2026-09-25 14:00:00,2706,154.65,200.0,"300,200",19806.43,EURUSD,70.29
80,evt_1745197200_81,2025-04-21 01:00:00,2025-06-20 00:00:00,86340,155.59,200.0,"300,200",9010.45,EURUSD,41.89
100,evt_1753781700_101,2025-07-29 09:35:00,2025-08-06 14:15:00,11800,156.70,200.0,"300,200",8772.43,EURUSD,64.81
53,evt_1715798700_54,2024-05-15 18:45:00,2024-05-16 11:15:00,990,157.70,200.0,"300,200",5471.07,EURUSD,52.95
71,evt_1728055860_72,2024-10-04 15:31:00,2024-10-07 20:30:00,4619,159.60,200.0,"300,200",7224.49,EURUSD,83.87
104,evt_1761921720_105,2025-10-31 14:42:00,2025-11-07 15:15:00,10113,160.57,200.0,"300,200",11835.27,EURUSD,51.22
106,evt_1770224402_107,2026-02-04 17:00:02,2026-02-05 15:30:00,1349,164.13,200.0,"300,200",14092.66,EURUSD,59.98



Basket Composition at 3 Worst Events


,event_id,worst_threshold_crossed,thresholds_crossed,min_margin_time,pair,open_positions,used_margin,floating_pnl,used_margin_contribution_pct,portfolio_used_margin,portfolio_margin_level
0,evt_1712153520_47,150.0,"300,200,150",2024-04-04 16:30:00,CADJPY,3,1483.42,-1116.54,25.51,5815.07,137.43
1,evt_1712153520_47,150.0,"300,200,150",2024-04-04 16:30:00,EURUSD,4,2935.76,-479.68,50.49,5815.07,137.43
2,evt_1712153520_47,150.0,"300,200,150",2024-04-04 16:30:00,NZDCHF,4,1395.89,-929.45,24.00,5815.07,137.43
3,evt_1715920200_55,150.0,"300,200,150",2024-05-30 05:00:00,CADJPY,3,1457.82,-1669.78,27.63,5277.13,148.21
4,evt_1715920200_55,150.0,"300,200,150",2024-05-30 05:00:00,EURUSD,4,1986.92,-99.66,37.65,5277.13,148.21
5,evt_1715920200_55,150.0,"300,200,150",2024-05-30 05:00:00,NZDCHF,5,1832.39,-1081.60,34.72,5277.13,148.21
6,evt_1702996860_10,200.0,"300,200",2023-12-20 17:45:00,CADJPY,2,975.90,-168.29,17.46,5589.94,154.40
7,evt_1702996860_10,200.0,"300,200",2023-12-20 17:45:00,EURUSD,4,3972.59,-673.52,71.07,5589.94,154.40
8,evt_1702996860_10,200.0,"300,200",2023-12-20 17:45:00,NZDCHF,2,641.45,-88.65,11.48,5589.94,154.40


## Margin Analysis

In [9]:
fig_equity = baseline.plot_equity()
fig_pair_balance = baseline.plot_pair_contributions()
#fig_pair_floating = baseline.plot_pair_floating()
fig_pair_drawdown = baseline.plot_pair_drawdown()
fig_margin = baseline.plot_margin()
fig_monthly_growth = baseline.plot_monthly_growth() 

fig_equity.show()
fig_pair_balance.show()
#fig_pair_floating.show()
fig_pair_drawdown.show()
fig_margin.show()
fig_monthly_growth.show()

## Swap Reports
Focused swap diagnostics for the baseline replay.

In [10]:
import os
import pandas as pd
from IPython.display import display

# Clean up stale swap_position_summary.csv from previous runs with different pairs
pos_path = os.path.join(REPO, 'output', 'swap_position_summary.csv')
if os.path.exists(pos_path):
    os.remove(pos_path)
    print(f'Removed stale {pos_path} to prevent displaying outdated position data.')

replay_df = baseline.replay_table().copy()
replay_df['time_dt'] = pd.to_datetime(replay_df['time'], format='%Y.%m.%d %H:%M:%S')
swap_df = replay_df[replay_df['EventType'] == 'swap'].copy()

if swap_df.empty:
    print('No swap rows were found in the current baseline replay output.')
else:
    swap_df['post_date'] = swap_df['time_dt'].dt.date
    swap_df['month'] = swap_df['time_dt'].dt.to_period('M').astype(str)

    total_swap = float(swap_df['ModeledSwap'].sum())
    swap_events = int(len(swap_df))
    avg_swap_per_event = float(total_swap / swap_events)
    avg_daily_swap = float(swap_df.groupby('post_date')['ModeledSwap'].sum().mean())

    deal_df = replay_df[replay_df['EventType'].astype(str).str.startswith('deal_')].copy()
    realized_profit = float(deal_df['scaled_net_profit'].sum())
    swap_pct_realized = (total_swap / realized_profit * 100.0) if realized_profit != 0 else float('nan')

    print('Swap summary')
    display(pd.DataFrame([
        {'Metric': 'Total modeled swap', 'Value': total_swap},
        {'Metric': 'Swap events', 'Value': swap_events},
        {'Metric': 'Average swap per event', 'Value': avg_swap_per_event},
        {'Metric': 'Average daily swap', 'Value': avg_daily_swap},
        {'Metric': 'Swap as % of realized profit', 'Value': swap_pct_realized},
    ]))

    print('Swap by pair')
    display(swap_df.groupby('pair', as_index=False)['ModeledSwap'].sum().sort_values('ModeledSwap'))

    print('Swap by direction')
    display(swap_df.groupby('direction', as_index=False)['ModeledSwap'].sum().sort_values('ModeledSwap'))

    print('Monthly swap')
    display(swap_df.groupby('month', as_index=False)['ModeledSwap'].sum().sort_values('month'))

    print('Top 20 positions by total swap')
    if os.path.exists(pos_path):
        pos_df = pd.read_csv(pos_path)
        display(pos_df.sort_values('Total Swap').head(20))
    else:
        print('swap_position_summary.csv not found in output/. Run swap_audit_report.py to generate it.')

    print('Cumulative swap over time (last 20 days)')
    daily = swap_df.groupby('post_date', as_index=False)['ModeledSwap'].sum().sort_values('post_date')
    daily['CumulativeSwap'] = daily['ModeledSwap'].cumsum()
    display(daily.tail(20))


Swap summary


,Metric,Value
0,Total modeled swap,-7055.080000
1,Swap events,6452.000000
2,Average swap per event,-1.093472
3,Average daily swap,-5.116084
4,Swap as % of realized profit,-19.396980


Swap by pair


,pair,ModeledSwap
2,NZDCHF,-3652.17
1,EURUSD,-1983.12
0,CADJPY,-1419.79


Swap by direction


,direction,ModeledSwap
1,sell,-5755.56
0,buy,-1299.52


Monthly swap


,month,ModeledSwap
0,2023-01,-9.19
1,2023-02,-16.24
2,2023-03,1.91
3,2023-04,15.62
4,2023-05,-4.27
5,2023-06,-38.82
6,2023-07,-36.35
7,2023-08,12.92
8,2023-09,-36.89
9,2023-10,-87.37


Top 20 positions by total swap
swap_position_summary.csv not found in output/. Run swap_audit_report.py to generate it.
Cumulative swap over time (last 20 days)


,post_date,ModeledSwap,CumulativeSwap
1359,2026-10-17,-4.57,-6855.95
1360,2026-10-18,-4.57,-6860.52
1361,2026-10-19,-4.57,-6865.09
1362,2026-10-20,-4.57,-6869.66
1363,2026-10-21,-11.85,-6881.51
1364,2026-10-22,-14.75,-6896.26
1365,2026-10-23,-7.33,-6903.59
1366,2026-10-24,-15.05,-6918.64
1367,2026-10-25,-15.05,-6933.69
1368,2026-10-26,-15.05,-6948.74
